In [1]:
# 运行这个单元格安装依赖（大约需要 1~2 分钟）
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # 最大上下文长度
dtype = None # 自动检测
load_in_4bit = True # 开启 4bit，显存立省 70%

# 加载 Gemma 4 E2B 对话模型
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 配置 LoRA 微调参数（只训练一小部分权重）
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj","gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [3]:
from datasets import load_dataset, concatenate_datasets

# 1. 下载通用中文指令数据 (Alpaca-zh, 2000条，保证日常对话和基础情商)
print("正在下载通用中文指令数据 (Alpaca-zh, 2000条)...")
alpaca_ds = load_dataset("shibing624/alpaca-zh", split="train[:2000]")

# 2. 下载 DeepSeek-R1 官方蒸馏中文深度思维链数据 (1000条，注入深度慢思考灵魂)
print("正在下载 DeepSeek-R1 中文深度思维链数据...")
r1_ds = load_dataset("Congliu/Chinese-DeepSeek-R1-Distill-data-110k-SFT", split="train[:1500]")

# 过滤超长文本（保留 2000 字符内，保证 T4 显存绝对安全）
r1_ds = r1_ds.filter(lambda x: len(x.get("output", "")) <= 2000 and len(x.get("output", "")) >= 80)
r1_ds = r1_ds.select(range(min(1000, len(r1_ds))))

# 3. 转换为 Gemma 官方标准模板
def format_alpaca(batch):
    texts = []
    for inst, inp, out in zip(batch["instruction"], batch["input"], batch["output"]):
        user_msg = f"{inst}\n{inp}".strip() if inp else inst
        text = f"<start_of_turn>user\n{user_msg}<end_of_turn>\n<start_of_turn>model\n{out}<end_of_turn>"
        texts.append(text)
    return {"text": texts}

def format_r1(batch):
    texts = []
    for inst, inp, out in zip(batch["instruction"], batch["input"], batch["output"]):
        user_msg = f"{inst}\n{inp}".strip() if inp else inst
        # out 已自带的 <think>...</think> 思考过程和结论
        text = f"<start_of_turn>user\n{user_msg}<end_of_turn>\n<start_of_turn>model\n{out}<end_of_turn>"
        texts.append(text)
    return {"text": texts}

alpaca_formatted = alpaca_ds.map(format_alpaca, batched=True, remove_columns=alpaca_ds.column_names)
r1_formatted = r1_ds.map(format_r1, batched=True, remove_columns=r1_ds.column_names)

# 4. 混合拼接（3000条：文武双全，既懂通用对话，又懂 R1 深度推演）
dataset = concatenate_datasets([alpaca_formatted, r1_formatted]).shuffle(seed=3407)

print(f"\n混合数据集准备完毕！总数据量: {len(dataset)} 条 (2000 通用 + 1000 R1 思维链)")
print("展示一条 R1 样本片段:\n", [x for x in dataset["text"] if "<think>" in x][0][:300], "...")

In [4]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 1, # 跑满 1 个 Epoch（约 375 步）
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        output_dir = "outputs",
    ),
)

# 开始训练
trainer.train()

In [6]:
# 1. 切换到推理加速模式
FastLanguageModel.for_inference(model)

# 2. 准备测试问题列表（包含经典的思维链陷阱题）
test_questions = [
    "9.11 和 9.9 哪个数字更大？",
]

print("🤖 开始检验 Gemma 4 + DeepSeek-R1 灵魂学习成果：\n" + "="*50)

for idx, q in enumerate(test_questions, 1):
    prompt = f"<start_of_turn>user\n{q}<end_of_turn>\n<start_of_turn>model\n"
    inputs = tokenizer(text = [prompt], return_tensors = "pt").to("cuda")
    
    # 留出充足草稿空间（max_new_tokens=512）
    outputs = model.generate(
        **inputs, 
        max_new_tokens = 2048, 
        use_cache = True,
        temperature = 0.6,
    )
    
    reply = tokenizer.batch_decode(outputs)[0]
    answer = reply.split("<start_of_turn>model\n")[-1].replace("<end_of_turn>", "")
    
    print(f"\n【问题 {idx}】：{q}")
    print(f"【Gemma 4 (R1-Distill) 回答】：\n{answer.strip()}")
    print("-" * 50)

In [7]:
import shutil
from google.colab import files

# 保存并打包专属的 Gemma-4-DeepSeek-R1-Distill 适配器权重
print("正在保存专属 R1 思维链 LoRA 适配器...")
model.save_pretrained("my_gemma_r1_lora")
tokenizer.save_pretrained("my_gemma_r1_lora")

print("正在打包为 zip 文件...")
shutil.make_archive("my_gemma_r1_lora", 'zip', "my_gemma_r1_lora")

print("打包成功，开始触发下载！")
files.download("my_gemma_r1_lora.zip")